In [1]:
from pathlib import Path
from collections import Counter
import json
import sys

import numpy as np
import pandas as pd
from datasets import load_dataset


# تحديد مجلد المشروع سواء النوتبوك مفتوحة من root أو notebooks
current_path = Path.cwd().resolve()

if (current_path / "src").exists():
    project_root = current_path
elif (current_path.parent / "src").exists():
    project_root = current_path.parent
else:
    raise FileNotFoundError(
        "Could not locate the project root folder."
    )

# إضافة مسار المشروع إلى Python path
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


# تحميل CoNLL-2003
dataset = load_dataset("tomaarsen/conll2003")

# استخراج أسماء الـNER labels
label_names = (
    dataset["train"]
    .features["ner_tags"]
    .feature.names
)

print(f"Current path: {current_path}")
print(f"Project root: {project_root}")
print()
print(dataset)
print()
print(f"NER labels: {label_names}")

Current path: D:\named_entity_recognition_project\notebooks
Project root: D:\named_entity_recognition_project

DatasetDict({
    train: Dataset({
        features: ['id', 'document_id', 'sentence_id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'document_id', 'sentence_id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'document_id', 'sentence_id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

NER labels: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']


In [2]:
token_character_lengths = [
    len(token)
    for sentence in dataset["train"]["tokens"]
    for token in sentence
]

character_length_statistics = pd.DataFrame({
    "percentile": [50, 75, 90, 95, 99, 99.5, 100],
    "token_length": np.percentile(
        token_character_lengths,
        [50, 75, 90, 95, 99, 99.5, 100]
    ).astype(int)
})

character_length_statistics

,percentile,token_length
0,50.0,4
1,75.0,6
2,90.0,8
3,95.0,9
4,99.0,12
5,99.5,13
6,100.0,61


In [3]:
print(
    f"Total training tokens: "
    f"{len(token_character_lengths):,}"
)

print(
    f"Average token length: "
    f"{np.mean(token_character_lengths):.2f}"
)

print(
    f"Maximum token length: "
    f"{max(token_character_lengths)}"
)

Total training tokens: 203,621
Average token length: 4.37
Maximum token length: 61


In [4]:
MAX_SEQUENCE_LENGTH = 128
MAX_WORD_LENGTH = 16

PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"

PAD_WORD_ID = 0
UNK_WORD_ID = 1

PAD_CHAR = "<PAD>"
UNK_CHAR = "<UNK>"

PAD_CHAR_ID = 0
UNK_CHAR_ID = 1

print(f"Maximum sentence length: {MAX_SEQUENCE_LENGTH}")
print(f"Maximum word length: {MAX_WORD_LENGTH}")

Maximum sentence length: 128
Maximum word length: 16


In [5]:
word_counter = Counter(
    token
    for sentence in dataset["train"]["tokens"]
    for token in sentence
)

word2idx = {
    PAD_TOKEN: PAD_WORD_ID,
    UNK_TOKEN: UNK_WORD_ID
}

# ترتيب الكلمات حسب التكرار، ثم أبجديًا لضمان نفس النتائج كل مرة
sorted_words = sorted(
    word_counter.items(),
    key=lambda item: (-item[1], item[0])
)

for word, frequency in sorted_words:
    word2idx[word] = len(word2idx)

idx2word = {
    index: word
    for word, index in word2idx.items()
}

print(f"Original training vocabulary: {len(word_counter):,}")
print(f"Final vocabulary size: {len(word2idx):,}")
print(f"PAD ID: {word2idx[PAD_TOKEN]}")
print(f"UNK ID: {word2idx[UNK_TOKEN]}")

Original training vocabulary: 23,623
Final vocabulary size: 23,625
PAD ID: 0
UNK ID: 1


In [6]:
character_counter = Counter(
    character
    for sentence in dataset["train"]["tokens"]
    for token in sentence
    for character in token
)

char2idx = {
    PAD_CHAR: PAD_CHAR_ID,
    UNK_CHAR: UNK_CHAR_ID
}

sorted_characters = sorted(
    character_counter.items(),
    key=lambda item: (-item[1], item[0])
)

for character, frequency in sorted_characters:
    char2idx[character] = len(char2idx)

idx2char = {
    index: character
    for character, index in char2idx.items()
}

print(f"Original character vocabulary: {len(character_counter):,}")
print(f"Final character vocabulary: {len(char2idx):,}")
print(f"PAD character ID: {char2idx[PAD_CHAR]}")
print(f"UNK character ID: {char2idx[UNK_CHAR]}")

Original character vocabulary: 84
Final character vocabulary: 86
PAD character ID: 0
UNK character ID: 1


In [7]:
label2idx = {
    label: index
    for index, label in enumerate(label_names)
}

idx2label = {
    index: label
    for label, index in label2idx.items()
}

print("Label mapping:")

for label_id, label_name in idx2label.items():
    print(f"{label_id}: {label_name}")

Label mapping:
0: O
1: B-PER
2: I-PER
3: B-ORG
4: I-ORG
5: B-LOC
6: I-LOC
7: B-MISC
8: I-MISC


In [8]:
sample = dataset["train"][0]

sample_tokens = sample["tokens"]
sample_label_ids = sample["ner_tags"]

sample_word_ids = [
    word2idx.get(token, UNK_WORD_ID)
    for token in sample_tokens
]

sample_character_ids = [
    [
        char2idx.get(character, UNK_CHAR_ID)
        for character in token[:MAX_WORD_LENGTH]
    ]
    for token in sample_tokens
]

sample_labels = [
    idx2label[label_id]
    for label_id in sample_label_ids
]

print("Sentence:")
print(" ".join(sample_tokens))

print("\nTokens:")
print(sample_tokens)

print("\nWord IDs:")
print(sample_word_ids)

print("\nNER Labels:")
print(sample_labels)

print("\nCharacter IDs:")
for token, character_ids in zip(
    sample_tokens,
    sample_character_ids
):
    print(f"{token:15} -> {character_ids}")

Sentence:
EU rejects German call to boycott British lamb .

Tokens:
['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.']

Word IDs:
[964, 22407, 236, 771, 7, 4586, 210, 7684, 2]

NER Labels:
['B-ORG', 'O', 'B-MISC', 'O', 'O', 'O', 'B-MISC', 'O', 'O']

Character IDs:
EU              -> [37, 59]
rejects         -> [8, 2, 63, 2, 13, 4, 9]
German          -> [53, 2, 8, 15, 3, 5]
call            -> [13, 3, 10, 10]
to              -> [4, 7]
boycott         -> [22, 7, 20, 13, 7, 4, 4]
British         -> [45, 8, 6, 4, 6, 9, 12]
lamb            -> [10, 3, 15, 22]
.               -> [19]


In [9]:
decoded_words = [
    idx2word[word_id]
    for word_id in sample_word_ids
]

decoded_characters_first_token = "".join(
    idx2char[character_id]
    for character_id in sample_character_ids[0]
)

print("Original tokens:")
print(sample_tokens)

print("\nDecoded tokens:")
print(decoded_words)

print("\nFirst token:")
print(sample_tokens[0])

print("Decoded first token from character IDs:")
print(decoded_characters_first_token)

Original tokens:
['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.']

Decoded tokens:
['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.']

First token:
EU
Decoded first token from character IDs:
EU


In [10]:
test_unknown_word = "ChatGPTCompany"

unknown_word_id = word2idx.get(
    test_unknown_word,
    UNK_WORD_ID
)

unknown_character_ids = [
    char2idx.get(character, UNK_CHAR_ID)
    for character in test_unknown_word[:MAX_WORD_LENGTH]
]

decoded_unknown_characters = "".join(
    idx2char.get(character_id, UNK_CHAR)
    for character_id in unknown_character_ids
)

print(f"Word: {test_unknown_word}")
print(f"Word ID: {unknown_word_id}")
print(f"Expected UNK ID: {UNK_WORD_ID}")
print(f"Character IDs: {unknown_character_ids}")
print(f"Decoded characters: {decoded_unknown_characters}")

Word: ChatGPTCompany
Word ID: 1
Expected UNK ID: 1
Character IDs: [35, 12, 3, 4, 53, 51, 32, 35, 7, 15, 17, 3, 5, 20]
Decoded characters: ChatGPTCompany


In [11]:
LABEL_PAD_ID = -100

print(f"Word padding ID: {PAD_WORD_ID}")
print(f"Character padding ID: {PAD_CHAR_ID}")
print(f"Label padding ID: {LABEL_PAD_ID}")

Word padding ID: 0
Character padding ID: 0
Label padding ID: -100


In [12]:
def encode_sample(
    tokens,
    label_ids,
    word2idx,
    char2idx,
    max_sequence_length=MAX_SEQUENCE_LENGTH,
    max_word_length=MAX_WORD_LENGTH
):
    """
    Convert one NER sample into fixed-size numerical arrays.
    """

    if len(tokens) != len(label_ids):
        raise ValueError(
            "Number of tokens must equal number of labels."
        )

    # Truncation
    tokens = tokens[:max_sequence_length]
    label_ids = label_ids[:max_sequence_length]

    sequence_length = len(tokens)
    padding_length = max_sequence_length - sequence_length

    # Word IDs
    word_ids = [
        word2idx.get(token, UNK_WORD_ID)
        for token in tokens
    ]

    # Character IDs
    character_ids = []

    for token in tokens:
        token_character_ids = [
            char2idx.get(character, UNK_CHAR_ID)
            for character in token[:max_word_length]
        ]

        character_padding_length = (
            max_word_length - len(token_character_ids)
        )

        token_character_ids += (
            [PAD_CHAR_ID] * character_padding_length
        )

        character_ids.append(token_character_ids)

    # 1 للكلمات الحقيقية، و0 للـPadding
    attention_mask = [1] * sequence_length

    # Sentence padding
    word_ids += [PAD_WORD_ID] * padding_length
    label_ids = list(label_ids) + [LABEL_PAD_ID] * padding_length
    attention_mask += [0] * padding_length

    character_ids += [
        [PAD_CHAR_ID] * max_word_length
        for _ in range(padding_length)
    ]

    return {
        "word_ids": np.asarray(word_ids, dtype=np.int64),
        "character_ids": np.asarray(
            character_ids,
            dtype=np.int64
        ),
        "labels": np.asarray(label_ids, dtype=np.int64),
        "attention_mask": np.asarray(
            attention_mask,
            dtype=np.bool_
        ),
        "sequence_length": sequence_length,
        "tokens": tokens
    }

In [13]:
encoded_sample = encode_sample(
    tokens=sample_tokens,
    label_ids=sample_label_ids,
    word2idx=word2idx,
    char2idx=char2idx
)

print("Word IDs shape:", encoded_sample["word_ids"].shape)
print(
    "Character IDs shape:",
    encoded_sample["character_ids"].shape
)
print("Labels shape:", encoded_sample["labels"].shape)
print(
    "Attention mask shape:",
    encoded_sample["attention_mask"].shape
)

print(
    "\nOriginal sequence length:",
    encoded_sample["sequence_length"]
)

print(
    "Valid tokens in mask:",
    encoded_sample["attention_mask"].sum()
)

Word IDs shape: (128,)
Character IDs shape: (128, 16)
Labels shape: (128,)
Attention mask shape: (128,)

Original sequence length: 9
Valid tokens in mask: 9


In [14]:
print("First 12 Word IDs:")
print(encoded_sample["word_ids"][:12])

print("\nFirst 12 Labels:")
print(encoded_sample["labels"][:12])

print("\nFirst 12 Mask values:")
print(encoded_sample["attention_mask"][:12].astype(int))

print("\nLast 5 Word IDs:")
print(encoded_sample["word_ids"][-5:])

print("\nLast 5 Labels:")
print(encoded_sample["labels"][-5:])

print("\nLast 5 Mask values:")
print(encoded_sample["attention_mask"][-5:].astype(int))

First 12 Word IDs:
[  964 22407   236   771     7  4586   210  7684     2     0     0     0]

First 12 Labels:
[   3    0    7    0    0    0    7    0    0 -100 -100 -100]

First 12 Mask values:
[1 1 1 1 1 1 1 1 1 0 0 0]

Last 5 Word IDs:
[0 0 0 0 0]

Last 5 Labels:
[-100 -100 -100 -100 -100]

Last 5 Mask values:
[0 0 0 0 0]


In [15]:
import torch
from torch.utils.data import Dataset, DataLoader


class NERDataset(Dataset):
    """
    PyTorch Dataset for CoNLL-2003 NER samples.
    """

    def __init__(
        self,
        dataset_split,
        word2idx,
        char2idx,
        max_sequence_length=MAX_SEQUENCE_LENGTH,
        max_word_length=MAX_WORD_LENGTH
    ):
        self.dataset_split = dataset_split
        self.word2idx = word2idx
        self.char2idx = char2idx
        self.max_sequence_length = max_sequence_length
        self.max_word_length = max_word_length

    def __len__(self):
        return len(self.dataset_split)

    def __getitem__(self, index):
        sample = self.dataset_split[index]

        encoded = encode_sample(
            tokens=sample["tokens"],
            label_ids=sample["ner_tags"],
            word2idx=self.word2idx,
            char2idx=self.char2idx,
            max_sequence_length=self.max_sequence_length,
            max_word_length=self.max_word_length
        )

        return {
            "word_ids": torch.tensor(
                encoded["word_ids"],
                dtype=torch.long
            ),
            "character_ids": torch.tensor(
                encoded["character_ids"],
                dtype=torch.long
            ),
            "labels": torch.tensor(
                encoded["labels"],
                dtype=torch.long
            ),
            "attention_mask": torch.tensor(
                encoded["attention_mask"],
                dtype=torch.bool
            ),
            "sequence_length": torch.tensor(
                encoded["sequence_length"],
                dtype=torch.long
            )
        }

In [16]:
train_dataset = NERDataset(
    dataset_split=dataset["train"],
    word2idx=word2idx,
    char2idx=char2idx
)

validation_dataset = NERDataset(
    dataset_split=dataset["validation"],
    word2idx=word2idx,
    char2idx=char2idx
)

test_dataset = NERDataset(
    dataset_split=dataset["test"],
    word2idx=word2idx,
    char2idx=char2idx
)

print(f"Train samples: {len(train_dataset):,}")
print(f"Validation samples: {len(validation_dataset):,}")
print(f"Test samples: {len(test_dataset):,}")

Train samples: 14,041
Validation samples: 3,250
Test samples: 3,453


In [17]:
first_dataset_sample = train_dataset[0]

for key, value in first_dataset_sample.items():
    print(
        f"{key:20} "
        f"shape={tuple(value.shape)}, "
        f"dtype={value.dtype}"
    )

word_ids             shape=(128,), dtype=torch.int64
character_ids        shape=(128, 16), dtype=torch.int64
labels               shape=(128,), dtype=torch.int64
attention_mask       shape=(128,), dtype=torch.bool
sequence_length      shape=(), dtype=torch.int64


In [18]:
BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print(f"Train batches: {len(train_loader):,}")
print(f"Validation batches: {len(validation_loader):,}")
print(f"Test batches: {len(test_loader):,}")

Train batches: 439
Validation batches: 102
Test batches: 108


In [19]:
first_batch = next(iter(train_loader))

print("Batch shapes:")

for key, value in first_batch.items():
    print(
        f"{key:20} "
        f"shape={tuple(value.shape)}, "
        f"dtype={value.dtype}"
    )

Batch shapes:
word_ids             shape=(32, 128), dtype=torch.int64
character_ids        shape=(32, 128, 16), dtype=torch.int64
labels               shape=(32, 128), dtype=torch.int64
attention_mask       shape=(32, 128), dtype=torch.bool
sequence_length      shape=(32,), dtype=torch.int64


In [20]:
processed_data_dir = project_root / "data" / "processed"
processed_data_dir.mkdir(parents=True, exist_ok=True)

mapping_files = {
    "word2idx.json": word2idx,
    "char2idx.json": char2idx,
    "label2idx.json": label2idx
}

for filename, mapping in mapping_files.items():
    file_path = processed_data_dir / filename

    with open(file_path, "w", encoding="utf-8") as file:
        json.dump(
            mapping,
            file,
            indent=4,
            ensure_ascii=False
        )

    print(f"Saved: {file_path}")

Saved: D:\named_entity_recognition_project\data\processed\word2idx.json
Saved: D:\named_entity_recognition_project\data\processed\char2idx.json
Saved: D:\named_entity_recognition_project\data\processed\label2idx.json


In [21]:
preprocessing_metadata = {
    "dataset_name": "CoNLL-2003",
    "dataset_source": "tomaarsen/conll2003",

    "max_sequence_length": MAX_SEQUENCE_LENGTH,
    "max_word_length": MAX_WORD_LENGTH,
    "batch_size": BATCH_SIZE,

    "word_vocabulary_size": len(word2idx),
    "character_vocabulary_size": len(char2idx),
    "number_of_labels": len(label2idx),

    "special_tokens": {
        "pad_token": PAD_TOKEN,
        "unk_token": UNK_TOKEN,
        "pad_word_id": PAD_WORD_ID,
        "unk_word_id": UNK_WORD_ID,

        "pad_character": PAD_CHAR,
        "unk_character": UNK_CHAR,
        "pad_character_id": PAD_CHAR_ID,
        "unk_character_id": UNK_CHAR_ID,

        "label_pad_id": LABEL_PAD_ID
    },

    "label_names": label_names,

    "dataset_sizes": {
        "train": len(train_dataset),
        "validation": len(validation_dataset),
        "test": len(test_dataset)
    },

    "batch_counts": {
        "train": len(train_loader),
        "validation": len(validation_loader),
        "test": len(test_loader)
    }
}

metadata_path = (
    processed_data_dir
    / "preprocessing_metadata.json"
)

with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(
        preprocessing_metadata,
        file,
        indent=4,
        ensure_ascii=False
    )

print(f"Saved metadata: {metadata_path}")

Saved metadata: D:\named_entity_recognition_project\data\processed\preprocessing_metadata.json


In [22]:
print("Processed data files:")

for file_path in sorted(processed_data_dir.iterdir()):
    print(
        f"{file_path.name:30} "
        f"{file_path.stat().st_size / 1024:.2f} KB"
    )

Processed data files:
char2idx.json                  1.18 KB
label2idx.json                 0.15 KB
preprocessing_metadata.json    0.96 KB
word2idx.json                  515.93 KB


In [23]:
with open(
    processed_data_dir / "word2idx.json",
    "r",
    encoding="utf-8"
) as file:
    loaded_word2idx = json.load(file)

with open(
    processed_data_dir / "char2idx.json",
    "r",
    encoding="utf-8"
) as file:
    loaded_char2idx = json.load(file)

with open(
    processed_data_dir / "label2idx.json",
    "r",
    encoding="utf-8"
) as file:
    loaded_label2idx = json.load(file)

with open(
    processed_data_dir / "preprocessing_metadata.json",
    "r",
    encoding="utf-8"
) as file:
    loaded_metadata = json.load(file)

print(
    "Loaded word vocabulary:",
    len(loaded_word2idx)
)

print(
    "Loaded character vocabulary:",
    len(loaded_char2idx)
)

print(
    "Loaded labels:",
    loaded_label2idx
)

print(
    "Maximum sequence length:",
    loaded_metadata["max_sequence_length"]
)

Loaded word vocabulary: 23625
Loaded character vocabulary: 86
Loaded labels: {'O': 0, 'B-PER': 1, 'I-PER': 2, 'B-ORG': 3, 'I-ORG': 4, 'B-LOC': 5, 'I-LOC': 6, 'B-MISC': 7, 'I-MISC': 8}
Maximum sequence length: 128


In [24]:
from src.data.dataset import NERDataset, encode_sample
from src.data.preprocess import load_json

test_word2idx = load_json(
    project_root / "data" / "processed" / "word2idx.json"
)

test_char2idx = load_json(
    project_root / "data" / "processed" / "char2idx.json"
)

test_dataset_from_src = NERDataset(
    dataset_split=dataset["train"],
    word2idx=test_word2idx,
    char2idx=test_char2idx
)

test_item = test_dataset_from_src[0]

for key, value in test_item.items():
    print(
        f"{key:20} "
        f"shape={tuple(value.shape)}, "
        f"dtype={value.dtype}"
    )

word_ids             shape=(128,), dtype=torch.int64
character_ids        shape=(128, 16), dtype=torch.int64
labels               shape=(128,), dtype=torch.int64
attention_mask       shape=(128,), dtype=torch.bool
sequence_length      shape=(), dtype=torch.int64
